# Analisis Pemodelan Karhutla DIMAS-TI 2026

Notebook ini menyajikan **metodologi dan hasil Orang B**. Training dan perhitungan SHAP tetap berada dalam skrip Python terpisah agar dapat direproduksi.

Urutan: dataset → perbandingan model → tuning → spatial validation → evaluasi → SHAP.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'models_final/model_spec_2026.json').exists()), None)
assert ROOT is not None, 'Buka notebook dari folder proyek karhutla'
spec = json.loads((ROOT / 'models_final/model_spec_2026.json').read_text(encoding='utf-8'))
print('Model:', spec['model'])
print('Target:', spec['target'])
print('Fitur:', spec['feature_order'])


## 1. Perbandingan LightGBM v2 dan v3

Feature engineering v3 menggunakan tanggal jeda sebagai sumber riwayat, tanpa menjadikannya baris training.

In [ ]:
path = ROOT / 'results_orang_b/comparison_v2_v3_validation_2026.csv'
display(pd.read_csv(path))


## 2. Hyperparameter tuning

Pemilihan kandidat dilakukan dengan **PR-AUC validation**, bukan berdasarkan data test.

In [ ]:
display(pd.read_csv(ROOT / 'results_orang_b/tuning_stage16_2026.csv'))
display(pd.read_csv(ROOT / 'results_orang_b/threshold_stage16_2026.csv'))


## 3. Spatial Validation

Membandingkan lokasi yang digunakan saat training dengan lokasi geografis yang ditahan dari training.

In [ ]:
display(pd.read_csv(ROOT / 'reports/orang_b/spatial_validation_2026.csv'))


## 4. Evaluasi Test September 2026

Hasil test ini **bukan untouched holdout**, karena September sudah pernah dilihat dalam eksperimen baseline.

In [ ]:
display(pd.read_csv(ROOT / 'reports/orang_b/stage19_metrics_2026.csv'))
display(Image(filename=str(ROOT / 'figures/orang_b/stage19_pr_curve_2026.png')))
display(Image(filename=str(ROOT / 'figures/orang_b/stage19_confusion_matrix_2026.png')))


## 5. Metode SHAP

Metode SHAP menjelaskan kontribusi fitur terhadap skor model. Pada LightGBM biner, kontribusi di sini menggunakan skala **log-odds**:

$$z=f_0+\sum_j\phi_j$$

$$P(y=1)=1/(1+e^{-z})$$

SHAP menjelaskan perilaku model, **bukan membuktikan hubungan sebab-akibat**.

In [ ]:
display(pd.read_csv(ROOT / 'reports/orang_b/stage20_shap_global_2026.csv'))
display(Image(filename=str(ROOT / 'figures/orang_b/stage20_shap_global_2026.png')))
display(Image(filename=str(ROOT / 'figures/orang_b/stage20_feature_importance_2026.png')))


## 6. Penjelasan Satu Prediksi dengan SHAP

SHAP positif meningkatkan log-odds hotspot menurut model. Nilai negatif menurunkannya.

In [ ]:
display(pd.read_csv(ROOT / 'reports/orang_b/stage20_shap_local_2026.csv'))
display(Image(filename=str(ROOT / 'figures/orang_b/stage20_shap_local_2026.png')))


## 7. Keterbatasan Penelitian

- Perubahan distribusi musiman antara train dan validation.
- Beberapa nilai ERA5 masih kosong.
- Keterlambatan publikasi GEE belum diaudit secara operasional.
- Generalisasi lintas tahun belum dibuktikan.
- Interpretasi SHAP bukan hubungan kausal.